# ETL IBGE — API SIDRA + PySpark

Notebook único do processo: descobre códigos e metadados da API, extrai os indicadores escolhidos, salva o JSON bruto, transforma os dados com PySpark e gera as saídas para o Power BI.

A divisão de responsabilidades é intencional:

- **Python + requests:** somente conexão HTTP e persistência do JSON bruto;
- **PySpark:** leitura, normalização, tipagem, tratamento de nulos, dimensões, cálculos e exportação.

## 1. Parâmetros do pipeline

Para adicionar um indicador, inclua uma entrada no dicionário `INDICADORES`. Os códigos podem ser descobertos na seção 3.

In [5]:
from pathlib import Path
from datetime import datetime
from urllib.parse import urlencode
from pyspark.sql import SparkSession, functions as F, types as T, Window

import json
import time
import requests

BASE_URL = 'https://servicodados.ibge.gov.br/api/v3/agregados'
BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name.lower() != 'ibge':
    BASE_DIR = BASE_DIR / 'IBGE'
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# False reutiliza o JSON mais recente; True consulta novamente a API.
ATUALIZAR_DADOS = True

IDADES_QUINQUENAIS = (
    '93070,93084,93085,93086,93087,93088,93089,93090,93091,93092,'
    '93093,93094,93095,93096,93097,93098,49108,49109,60040,60041,6653'
)
GRUPOS_IPCA = '7169,7170,7445,7486,7558,7625,7660,7712,7766,7786'

INDICADORES = {
    'populacao': {
        'agregado': '6579', 'periodos': '2012-2021', 'variaveis': '9324',
        'localidades': 'N6[all]', 'classificacao': None,
    },
    'pib': {
        'agregado': '6784', 'periodos': '2012-2021', 'variaveis': 'all',
        'localidades': 'N1[all]', 'classificacao': None,
    },
    'faixa_etaria': {
        'agregado': '9514', 'periodos': '2022', 'variaveis': '93',
        'localidades': 'N3[all]',
        'classificacao': f'2[6794,4,5]|287[{IDADES_QUINQUENAIS}]|286[113635]',
    },
    'desemprego': {
        'agregado': '4099', 'periodos': 'all', 'variaveis': '4099,4118',
        'localidades': 'N3[all]', 'classificacao': None,
    },
    'ipca': {
        'agregado': '7060', 'periodos': 'all', 'variaveis': '63,69,2265,66',
        'localidades': 'N7[all]',
        'classificacao': f'315[{GRUPOS_IPCA}]',
    },
}

# Remova nomes desta lista para executar apenas parte do pipeline.
INDICADORES_ATIVOS = ['populacao', 'pib', 'faixa_etaria', 'desemprego', 'ipca']
print('Pasta do projeto:', BASE_DIR)
print('Indicadores ativos:', INDICADORES_ATIVOS)

Pasta do projeto: C:\Users\Gabriel\Desktop\Python_Treinamento\projeto-indicadores-publicos\APIs\IBGE\src\IBGE
Indicadores ativos: ['populacao', 'pib', 'faixa_etaria', 'desemprego', 'ipca']


## 2. Sessão Spark e esquemas da API

In [6]:
spark = (
    SparkSession.builder
    .appName('ETL-IBGE-SIDRA')
    .master('local[*]')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')

categoria_schema = T.MapType(T.StringType(), T.StringType())
classificacao_schema = T.StructType([
    T.StructField('id', T.StringType()),
    T.StructField('nome', T.StringType()),
    T.StructField('categoria', categoria_schema),
])
localidade_schema = T.StructType([
    T.StructField('id', T.StringType()),
    T.StructField('nivel', T.StructType([
        T.StructField('id', T.StringType()),
        T.StructField('nome', T.StringType()),
    ])),
    T.StructField('nome', T.StringType()),
])
serie_schema = T.StructType([
    T.StructField('localidade', localidade_schema),
    T.StructField('serie', T.MapType(T.StringType(), T.StringType())),
])
resultado_schema = T.StructType([
    T.StructField('classificacoes', T.ArrayType(classificacao_schema)),
    T.StructField('series', T.ArrayType(serie_schema)),
])
variavel_schema = T.StructType([
    T.StructField('id', T.StringType()),
    T.StructField('variavel', T.StringType()),
    T.StructField('unidade', T.StringType()),
    T.StructField('resultados', T.ArrayType(resultado_schema)),
])
sidra_schema = T.ArrayType(variavel_schema)
spark

## 3. Descoberta dos códigos da API

Informe o código de uma tabela para visualizar variáveis, níveis territoriais e classificações. Esta etapa é exploratória e não precisa ser executada em toda atualização.

In [7]:
def consultar_api(url, tentativas=3, timeout=60):
    """Único ponto de conexão HTTP do notebook."""
    for tentativa in range(1, tentativas + 1):
        try:
            resposta = requests.get(url, timeout=timeout)
            resposta.raise_for_status()
            return resposta.json()
        except requests.RequestException:
            if tentativa == tentativas:
                raise
            time.sleep(3 * tentativa)

def descobrir_agregado(codigo):
    meta = consultar_api(f'{BASE_URL}/{codigo}/metadados')
    print(f"Tabela {codigo}: {meta['nome']}")
    print('\nNíveis territoriais:', meta.get('nivelTerritorial', {}))
    print('\nVariáveis:')
    for item in meta.get('variaveis', []):
        print(f"  {item['id']}: {item['nome']} [{item.get('unidade', '')}]")
    print('\nClassificações e categorias:')
    for classe in meta.get('classificacoes', []):
        print(f"  {classe['id']}: {classe['nome']}")
        for categoria in classe.get('categorias', []):
            print(f"    {categoria['id']}: {categoria['nome']} (nível {categoria['nivel']})")
    return meta

# Exemplos — descomente um de cada vez para explorar:
# metadados = descobrir_agregado('9514')   # faixa etária
# metadados = descobrir_agregado('4099')   # desemprego
# metadados = descobrir_agregado('7060')   # IPCA

## 4. Extração e camada bruta

A extração usa apenas Python/requests. Cada resposta é preservada em JSON para que as transformações possam ser refeitas sem consultar novamente a API.

In [ ]:
def montar_url(config):
    query = {'localidades': config['localidades']}
    if config.get('classificacao'):
        query['classificacao'] = config['classificacao']
    return (
        f"{BASE_URL}/{config['agregado']}/periodos/{config['periodos']}"
        f"/variaveis/{config['variaveis']}?{urlencode(query, safe='[],|')}"
    )

def extrair_indicador(nome, config):
    url = montar_url(config)
    print(f'Extraindo {nome}: {url}')
    dados = consultar_api(url)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    destino = RAW_DIR / f"ibge_{nome}_{config['agregado']}_{timestamp}.json"
    with destino.open('w', encoding='utf-8') as arquivo:
        json.dump(dados, arquivo, ensure_ascii=False, indent=2)
    print('Salvo:', destino)
    return destino

def json_mais_recente(nome, agregado):
    arquivos = sorted(RAW_DIR.glob(f'ibge_{nome}_{agregado}_*.json'))
    # Compatibilidade com os primeiros arquivos do projeto.
    if nome in {'populacao', 'pib'}:
        arquivos += sorted(RAW_DIR.glob(f'ibge_agregado_{agregado}_*.json'))
        arquivos = sorted(arquivos, key=lambda caminho: caminho.stat().st_mtime)
    if not arquivos:
        raise FileNotFoundError(f'Não existe JSON bruto para {nome}. Ative ATUALIZAR_DADOS.')
    return arquivos[-1]

arquivos_raw = {}
for nome in INDICADORES_ATIVOS:
    config = INDICADORES[nome]
    arquivos_raw[nome] = (
        extrair_indicador(nome, config)
        if ATUALIZAR_DADOS
        else json_mais_recente(nome, config['agregado'])
    )

arquivos_raw

Extraindo populacao: https://servicodados.ibge.gov.br/api/v3/agregados/6579/periodos/2012-2021/variaveis/9324?localidades=N6[all]


HTTPError: 500 Server Error: Internal Server Error for url: https://servicodados.ibge.gov.br/api/v3/agregados/6579/periodos/2012-2021/variaveis/9324?localidades=N6%5Ball%5D

## 5. Normalização genérica com PySpark

O JSON é lido como texto integral e interpretado com um esquema explícito. Séries, períodos e classificações são explodidos para o formato tabular longo.

## DF_POPULAÇÃO

In [ ]:
df_populacao = (
    spark.read
    .option("multiline", "true")
    .json(r"C:\Users\Gabriel\Desktop\Python_Treinamento\projeto-indicadores-publicos\APIs\IBGE\data\raw\ibge_agregado_6579_20260728_104316.json")
)
df_populacao = df_populacao.select(
            F.col('id').alias('variavel_id'),
            F.col('variavel').alias('variavel_nome'),
            F.col('unidade').alias('unidade'),
            F.explode('resultados').alias('resultado'),
        )
df_populacao = df_populacao.select("variavel_id", "variavel_nome", "unidade", F.explode("resultado.series").alias("serie_item"))

In [15]:
df_populacao = df_populacao.select("variavel_id", "variavel_nome", "unidade",
    F.col("serie_item.localidade.id").cast("int").alias("municipio_id"),
    F.col("serie_item.localidade.nome").alias("municipio_nome"),
    F.col("serie_item.localidade.nivel.id").alias("nivel_id"),
    F.col("serie_item.localidade.nivel.nome").alias("nivel_nome"),
    F.col("serie_item.serie").alias("serie")
)

In [16]:
anos = df_populacao.schema["serie"].dataType.fieldNames()

serie_map = F.create_map(*[elemento  for ano in anos
        for elemento in (F.lit(ano), F.col(f"serie.`{ano}`"))]
)

In [23]:
df_final_populacao = (df_populacao
    .select("variavel_id", "variavel_nome", "unidade", "municipio_id", "municipio_nome", "nivel_id", "nivel_nome",
        F.explode(serie_map).alias("ano", "populacao_original"))
    .withColumn("ano", F.col("ano").cast("int"))
    .withColumn("populacao", F.col("populacao_original").cast("long"))
    .withColumn("uf_id", F.substring(F.col("municipio_id").cast("string"), 1, 2).cast("int"))
    .withColumn("uf_sigla", F.regexp_extract(F.col("municipio_nome"), r"([A-Z]{2})$", 1))
    .withColumn("municipio_nome", F.regexp_replace(F.col("municipio_nome"), r"\s*-\s*[A-Z]{2}$", ""))
    .drop("populacao_original")
    )
df_final_populacao.orderBy("municipio_nome", "ano").show(20, truncate=False)
df_final_populacao.printSchema()

+-----------+----------------------------+-------+------------+-------------------+--------+----------+----+---------+-----+--------+
|variavel_id|variavel_nome               |unidade|municipio_id|municipio_nome     |nivel_id|nivel_nome|ano |populacao|uf_id|uf_sigla|
+-----------+----------------------------+-------+------------+-------------------+--------+----------+----+---------+-----+--------+
|9324       |População residente estimada|Pessoas|5200050     |Abadia de Goiás    |N6      |Município |2012|7164     |52   |GO      |
|9324       |População residente estimada|Pessoas|5200050     |Abadia de Goiás    |N6      |Município |2013|7567     |52   |GO      |
|9324       |População residente estimada|Pessoas|5200050     |Abadia de Goiás    |N6      |Município |2014|7733     |52   |GO      |
|9324       |População residente estimada|Pessoas|5200050     |Abadia de Goiás    |N6      |Município |2015|7895     |52   |GO      |
|9324       |População residente estimada|Pessoas|5200050     

In [24]:
caminho_saida = Path("../data/processed/populacao_municipio.csv")
caminho_saida.parent.mkdir(parents=True, exist_ok=True)

df_final_populacao.toPandas().to_csv(caminho_saida, index=False, encoding="utf-8-sig")

## DF_PIB

In [44]:
df_pib = (
    spark.read
    .option("multiline", "true")
    .json(r'C:\Users\Gabriel\Desktop\Python_Treinamento\projeto-indicadores-publicos\APIs\IBGE\data\raw\ibge_agregado_6784_20260728_104317.json')
)
df_pib = df_pib.select(
        F.col("id").cast("int").alias("variavel_id"),
        F.col("variavel").alias("variavel_nome"),
        F.col("unidade"),
        F.explode("resultados").alias("resultado")
    )

In [45]:
df_pib = (df_pib.select("variavel_id", "variavel_nome", "unidade", F.explode("resultado.series").alias("serie_item"))
    .select("variavel_id", "variavel_nome", "unidade",
        F.col("serie_item.localidade.id")
            .cast("int")
            .alias("pais_id"),
        F.col("serie_item.localidade.nome")
            .alias("pais_nome"),
        F.col("serie_item.localidade.nivel.id")
            .alias("nivel_id"),
        F.col("serie_item.localidade.nivel.nome")
            .alias("nivel_nome"),
        F.col("serie_item.serie").alias("serie")
    )
)

In [46]:
anos_pib = df_pib.schema["serie"].dataType.fieldNames()

serie_pib_map = F.create_map(
    *[
        elemento
        for ano in anos_pib
        for elemento in (
            F.lit(ano),
            F.col(f"serie.`{ano}`")
        )
    ]
)

In [50]:
df_final_pib = (df_pib.select(
        "variavel_id",
        "variavel_nome",
        "unidade",
        "pais_id",
        "pais_nome",
        "nivel_id",
        "nivel_nome",
        F.explode(serie_pib_map).alias("ano", "valor_original"))
    .withColumn("data_referencia", F.make_date(F.col("ano"), F.lit(1), F.lit(1)))
    .withColumn("ano", F.year("data_referencia"))
    .withColumn("valor", F.when(F.col("valor_original").rlike(r"^[-+]?\d+(?:[.,]\d+)?$"),
            F.regexp_replace(F.col("valor_original"), ",", ".").cast("double")))
    .drop("valor_original")
    .orderBy("variavel_id", "ano")
)

In [53]:
caminho_saida = Path("../data/processed")
caminho_saida.parent.mkdir(parents=True, exist_ok=True)

colunas_dimensao = [
    "variavel_id",
    "variavel_nome",
    "unidade",
    "pais_id",
    "pais_nome",
    "nivel_id",
    "nivel_nome",
    "data_referencia",
    "ano"
]

In [55]:
df_pib_pessoas = (
    df_final_pib
    .filter(F.col("variavel_id") == 93)
    .withColumn(
        "populacao_mil_pessoas",
        F.col("valor").cast("long")
    )
    .select(
        *colunas_dimensao,
        "populacao_mil_pessoas"
    )
    .orderBy("data_referencia")
)

df_pib_pessoas.toPandas().to_csv(
    caminho_saida / "PIB_pessoas.csv",
    index=False,
    encoding="utf-8-sig"
)

In [56]:
ids_financeiros = [9808, 9809, 9812, 9813]

df_pib_financeiro = (
    df_final_pib
    .filter(F.col("variavel_id").isin(ids_financeiros))
    .withColumn(
        "valor_financeiro",
        F.col("valor").cast("decimal(20, 2)")
    )
    .select(
        *colunas_dimensao,
        "valor_financeiro"
    )
    .orderBy("variavel_id", "data_referencia")
)

df_pib_financeiro.toPandas().to_csv(
    caminho_saida / "PIB_financeiro.csv",
    index=False,
    encoding="utf-8-sig"
)

In [57]:
ids_percentuais = [9810, 9811, 9814]

df_pib_percentual = (
    df_final_pib
    .filter(F.col("variavel_id").isin(ids_percentuais))
    .withColumn(
        "valor_percentual",
        F.col("valor").cast("decimal(10, 2)")
    )
    .select(
        *colunas_dimensao,
        "valor_percentual"
    )
    .orderBy("variavel_id", "data_referencia")
)

df_pib_percentual.toPandas().to_csv(
    caminho_saida / "PIB_percentual.csv",
    index=False,
    encoding="utf-8-sig"
)

In [51]:
df_final_pib.show(10, truncate=False)
df_final_pib.printSchema()

print("Quantidade de linhas:", df_final_pib.count())

+-----------+-------------------+-----------+-------+---------+--------+----------+----+---------------+--------+
|variavel_id|variavel_nome      |unidade    |pais_id|pais_nome|nivel_id|nivel_nome|ano |data_referencia|valor   |
+-----------+-------------------+-----------+-------+---------+--------+----------+----+---------------+--------+
|93         |População residente|Mil pessoas|1      |Brasil   |N1      |Brasil    |2012|2012-01-01     |197671.0|
|93         |População residente|Mil pessoas|1      |Brasil   |N1      |Brasil    |2013|2013-01-01     |199227.0|
|93         |População residente|Mil pessoas|1      |Brasil   |N1      |Brasil    |2014|2014-01-01     |200811.0|
|93         |População residente|Mil pessoas|1      |Brasil   |N1      |Brasil    |2015|2015-01-01     |202404.0|
|93         |População residente|Mil pessoas|1      |Brasil   |N1      |Brasil    |2016|2016-01-01     |203872.0|
|93         |População residente|Mil pessoas|1      |Brasil   |N1      |Brasil    |2017|

In [52]:
caminho_saida = Path("../data/processed/pib_brasil_ano.csv")
caminho_saida.parent.mkdir(parents=True, exist_ok=True)

df_final_pib.toPandas().to_csv(
    caminho_saida,
    index=False,
    encoding="utf-8-sig"
)

## DF_FAIXA_SALARIAL

In [70]:
df_faixa_raw = (
    spark.read
    .option("multiline", "true")
    .json(r'C:\Users\Gabriel\Desktop\Python_Treinamento\projeto-indicadores-publicos\APIs\IBGE\data\raw\ibge_faixa_etaria_9514_20260728_105028.json')
)

In [71]:
df_faixa_resultados = (
    df_faixa_raw
    .select(
        F.col("id").cast("int").alias("variavel_id"),
        F.col("variavel").alias("variavel_nome"),
        F.col("unidade"),
        F.explode("resultados").alias("resultado")
    )
)

In [72]:
df_faixa_series = (
    df_faixa_resultados
    .select(
        "variavel_id",
        "variavel_nome",
        "unidade",
        F.col("resultado.classificacoes").alias("classificacoes"),
        F.explode("resultado.series").alias("serie_item")
    )
)

In [73]:
mapa_categoria_schema = T.MapType(
    T.StringType(),
    T.StringType()
)

def extrair_categoria(classificacao_id):
    classificacao = F.element_at(
        F.filter(
            F.col("classificacoes"),
            lambda item: item["id"] == F.lit(str(classificacao_id))
        ),
        1
    )

    mapa = F.from_json(
        F.to_json(classificacao["categoria"]),
        mapa_categoria_schema
    )

    return F.element_at(F.map_entries(mapa), 1)

In [74]:
df_faixa_classificacoes = (
    df_faixa_series
    .withColumn(
        "categoria_sexo",
        extrair_categoria(2)
    )
    .withColumn(
        "categoria_idade",
        extrair_categoria(287)
    )
    .withColumn(
        "categoria_declaracao",
        extrair_categoria(286)
    )
)

In [75]:
mapa_serie_schema = T.MapType(
    T.StringType(),
    T.StringType()
)

df_faixa_explodido = (
    df_faixa_classificacoes
    .select(
        "variavel_id",
        "variavel_nome",
        "unidade",

        F.col("serie_item.localidade.id")
            .cast("int")
            .alias("uf_id"),

        F.col("serie_item.localidade.nome")
            .alias("uf_nome"),

        F.col("serie_item.localidade.nivel.id")
            .alias("nivel_id"),

        F.col("serie_item.localidade.nivel.nome")
            .alias("nivel_nome"),

        F.col("categoria_sexo.key")
            .cast("int")
            .alias("sexo_id"),

        F.col("categoria_sexo.value")
            .alias("sexo"),

        F.col("categoria_idade.key")
            .cast("int")
            .alias("faixa_etaria_id"),

        F.col("categoria_idade.value")
            .alias("faixa_etaria"),

        F.col("categoria_declaracao.key")
            .cast("int")
            .alias("declaracao_id"),

        F.col("categoria_declaracao.value")
            .alias("forma_declaracao"),

        F.explode(
            F.from_json(
                F.to_json("serie_item.serie"),
                mapa_serie_schema
            )
        ).alias("ano_original", "populacao_original")
    )
)

In [76]:
df_final_faixa_etaria = (
    df_faixa_explodido
    .withColumn(
        "ano",
        F.col("ano_original").cast("int")
    )
    .withColumn(
        "data_referencia",
        F.make_date(
            F.col("ano"),
            F.lit(1),
            F.lit(1)
        )
    )
    .withColumn(
        "populacao",
        F.when(
            F.col("populacao_original").rlike(r"^\d+$"),
            F.col("populacao_original").cast("long")
        )
    )
    .withColumn(
        "faixa_etaria_ordem",
        F.regexp_extract(
            F.col("faixa_etaria"),
            r"^(\d+)",
            1
        ).cast("int")
    )
    .drop(
        "ano_original",
        "populacao_original"
    )
)

In [77]:
df_final_faixa_etaria = (
    df_final_faixa_etaria
    .withColumn(
        "faixa_macro",
        F.when(
            F.col("faixa_etaria_ordem") <= 14,
            "0 a 14 anos"
        )
        .when(
            F.col("faixa_etaria_ordem") <= 29,
            "15 a 29 anos"
        )
        .when(
            F.col("faixa_etaria_ordem") <= 59,
            "30 a 59 anos"
        )
        .otherwise("60 anos ou mais")
    )
    .select(
        "variavel_id",
        "variavel_nome",
        "unidade",
        "data_referencia",
        "ano",
        "uf_id",
        "uf_nome",
        "nivel_id",
        "nivel_nome",
        "sexo_id",
        "sexo",
        "faixa_etaria_id",
        "faixa_etaria",
        "faixa_etaria_ordem",
        "faixa_macro",
        "populacao"
    )
    .orderBy(
        "uf_id",
        "sexo_id",
        "faixa_etaria_ordem"
    )
)

In [78]:
df_final_faixa_etaria.show(30, truncate=False)
df_final_faixa_etaria.printSchema()

print(
    "Quantidade de linhas:",
    df_final_faixa_etaria.count()
)

+-----------+-------------------+-------+---------------+----+-----+--------+--------+--------------------+-------+--------+---------------+----------------+------------------+---------------+---------+
|variavel_id|variavel_nome      |unidade|data_referencia|ano |uf_id|uf_nome |nivel_id|nivel_nome          |sexo_id|sexo    |faixa_etaria_id|faixa_etaria    |faixa_etaria_ordem|faixa_macro    |populacao|
+-----------+-------------------+-------+---------------+----+-----+--------+--------+--------------------+-------+--------+---------------+----------------+------------------+---------------+---------+
|93         |População residente|Pessoas|2022-01-01     |2022|11   |Rondônia|N3      |Unidade da Federação|4      |Homens  |93070          |0 a 4 anos      |0                 |0 a 14 anos    |57661    |
|93         |População residente|Pessoas|2022-01-01     |2022|11   |Rondônia|N3      |Unidade da Federação|4      |Homens  |93084          |5 a 9 anos      |5                 |0 a 14 anos 

In [79]:
caminho_saida = Path("../data/processed/faixa_etaria_uf.csv")

df_final_faixa_etaria.toPandas().to_csv(
    caminho_saida,
    index=False,
    encoding="utf-8-sig"
)

## DF_DESEMPREGO

In [61]:
df_desemprego = (
    spark.read
    .option("multiline", "true")
    .json(r'C:\Users\Gabriel\Desktop\Python_Treinamento\projeto-indicadores-publicos\APIs\IBGE\data\raw\ibge_desemprego_4099_20260728_105032.json')
)

## DF_IPCA

In [62]:
df_ipca = (
    spark.read
    .option("multiline", "true")
    .json(r'C:\Users\Gabriel\Desktop\Python_Treinamento\projeto-indicadores-publicos\APIs\IBGE\data\raw\ibge_ipca_7060_20260728_105037.json')
)

## 6. Dimensões compartilhadas

In [59]:
caminho_saida = Path("../data/processed/dim_uf.csv")
caminho_saida.parent.mkdir(parents=True, exist_ok=True)

ufs = [
    (11,'RO','Rondônia','Norte'),(12,'AC','Acre','Norte'),(13,'AM','Amazonas','Norte'),
    (14,'RR','Roraima','Norte'),(15,'PA','Pará','Norte'),(16,'AP','Amapá','Norte'),
    (17,'TO','Tocantins','Norte'),(21,'MA','Maranhão','Nordeste'),(22,'PI','Piauí','Nordeste'),
    (23,'CE','Ceará','Nordeste'),(24,'RN','Rio Grande do Norte','Nordeste'),
    (25,'PB','Paraíba','Nordeste'),(26,'PE','Pernambuco','Nordeste'),(27,'AL','Alagoas','Nordeste'),
    (28,'SE','Sergipe','Nordeste'),(29,'BA','Bahia','Nordeste'),(31,'MG','Minas Gerais','Sudeste'),
    (32,'ES','Espírito Santo','Sudeste'),(33,'RJ','Rio de Janeiro','Sudeste'),
    (35,'SP','São Paulo','Sudeste'),(41,'PR','Paraná','Sul'),(42,'SC','Santa Catarina','Sul'),
    (43,'RS','Rio Grande do Sul','Sul'),(50,'MS','Mato Grosso do Sul','Centro-Oeste'),
    (51,'MT','Mato Grosso','Centro-Oeste'),(52,'GO','Goiás','Centro-Oeste'),
    (53,'DF','Distrito Federal','Centro-Oeste'),
]
dim_uf = spark.createDataFrame(ufs, ['uf_id', 'uf_sigla', 'uf_nome', 'regiao'])
dim_uf.orderBy('uf_id')

dim_uf.toPandas().to_csv(
    caminho_saida,
    index=False,
    encoding="utf-8-sig"
)

## ANTIGO
    ## 7. Tratamentos por indicador
    ## 8. Qualidade e encerramento

In [ ]:
def ler_sidra(nome):
    caminho = str(arquivos_raw[nome])
    return (
        spark.read.text(caminho, wholetext=True)
        .select(F.explode(F.from_json('value', sidra_schema)).alias('variavel'))
        .select(
            F.col('variavel.id').alias('variavel_id'),
            F.col('variavel.variavel').alias('variavel_nome'),
            F.col('variavel.unidade').alias('unidade'),
            F.explode('variavel.resultados').alias('resultado'),
        )
        .select(
            'variavel_id', 'variavel_nome', 'unidade',
            F.col('resultado.classificacoes').alias('classificacoes'),
            F.explode('resultado.series').alias('serie'),
        )
        .select(
            'variavel_id', 'variavel_nome', 'unidade', 'classificacoes',
            F.col('serie.localidade.id').alias('localidade_id'),
            F.col('serie.localidade.nome').alias('localidade_nome'),
            F.col('serie.localidade.nivel.id').alias('nivel_id'),
            F.explode('serie.serie').alias('periodo', 'valor_original'),
        )
        .withColumn(
            'valor',
            F.when(
                F.col('valor_original').rlike(r'^[-+]?\d+(?:[.,]\d+)?$'),
                F.regexp_replace('valor_original', ',', '.').cast('double'),
            ),
        )
    )

def categoria(classificacao_id):
    encontrada = F.element_at(
        F.filter('classificacoes', lambda item: item['id'] == F.lit(str(classificacao_id))),
        1,
    )
    return F.element_at(F.map_values(encontrada['categoria']), 1)

def salvar_csv(df, nome):
    """Coleta somente a camada final, já tratada pelo Spark, para um CSV único."""
    destino = PROCESSED_DIR / f'{nome}.csv'
    df.toPandas().to_csv(destino, index=False, encoding='utf-8-sig')
    print(f'{nome}: {df.count():,} linhas -> {destino}')
    return destino

In [7]:
# População municipal anual
if 'populacao' in INDICADORES_ATIVOS:
    janela = Window.partitionBy('municipio_id').orderBy('ano')
    fato_populacao = (
        ler_sidra('populacao')
        .withColumn('municipio_id', F.col('localidade_id').cast('int'))
        .withColumn('municipio_nome', F.col('localidade_nome'))
        .withColumn('uf_id', F.substring('localidade_id', 1, 2).cast('int'))
        .withColumn('ano', F.col('periodo').cast('int'))
        .withColumn('populacao', F.col('valor').cast('long'))
        .withColumn('populacao_ano_anterior', F.lag('populacao').over(janela))
        .withColumn('variacao_absoluta', F.col('populacao') - F.col('populacao_ano_anterior'))
        .withColumn(
            'crescimento_pct',
            F.round(F.col('variacao_absoluta') / F.col('populacao_ano_anterior') * 100, 2),
        )
        .join(F.broadcast(dim_uf), 'uf_id', 'left')
        .select(
            'ano','municipio_id','municipio_nome','uf_id','uf_sigla','uf_nome','regiao',
            'populacao','populacao_ano_anterior','variacao_absoluta','crescimento_pct',
        )
        .orderBy('municipio_id', 'ano')
    )
    salvar_csv(fato_populacao, 'populacao_municipio')

populacao_municipio: 55,710 linhas -> C:\Users\Gabriel\Desktop\Python_Treinamento\IBGE\data\processed\populacao_municipio.csv


In [8]:
# PIB e contas nacionais no formato longo
if 'pib' in INDICADORES_ATIVOS:
    fato_pib = (
        ler_sidra('pib')
        .withColumn('ano', F.col('periodo').cast('int'))
        .select(
            'ano', 'localidade_id', 'localidade_nome', 'variavel_id',
            'variavel_nome', 'unidade', 'valor',
        )
        .orderBy('ano', 'variavel_id')
    )
    salvar_csv(fato_pib, 'pib_brasil_ano')

pib_brasil_ano: 80 linhas -> C:\Users\Gabriel\Desktop\Python_Treinamento\IBGE\data\processed\pib_brasil_ano.csv


In [9]:
# Faixa etária por UF e sexo — fotografia do Censo 2022
if 'faixa_etaria' in INDICADORES_ATIVOS:
    base_idade = (
        ler_sidra('faixa_etaria')
        .withColumn('sexo', categoria(2))
        .withColumn('faixa_etaria', categoria(287))
        .withColumn('uf_id', F.col('localidade_id').cast('int'))
        .withColumn('ano', F.col('periodo').cast('int'))
        .withColumn('populacao', F.col('valor').cast('long'))
    )
    inicio_idade = F.regexp_extract('faixa_etaria', r'^(\d+)', 1).cast('int')
    janela_idade = Window.partitionBy('uf_id', 'ano', 'sexo')
    fato_faixa_etaria = (
        base_idade
        .withColumn(
            'faixa_macro',
            F.when(inicio_idade <= 14, '0 a 14 anos')
             .when(inicio_idade <= 29, '15 a 29 anos')
             .when(inicio_idade <= 59, '30 a 59 anos')
             .otherwise('60 anos ou mais'),
        )
        .withColumn('populacao_base', F.sum('populacao').over(janela_idade))
        .withColumn('participacao_uf_pct', F.round(F.col('populacao') / F.col('populacao_base') * 100, 2))
        .join(F.broadcast(dim_uf), 'uf_id', 'left')
        .select(
            'ano','uf_id','uf_sigla','uf_nome','regiao','sexo','faixa_etaria',
            'faixa_macro','populacao','participacao_uf_pct',
        )
        .orderBy('uf_id', 'sexo', 'faixa_etaria')
    )
    salvar_csv(fato_faixa_etaria, 'faixa_etaria_uf')

faixa_etaria_uf: 1,701 linhas -> C:\Users\Gabriel\Desktop\Python_Treinamento\IBGE\data\processed\faixa_etaria_uf.csv


In [17]:
# Desemprego e subutilização trimestral por UF
if 'desemprego' in INDICADORES_ATIVOS:
    fato_desemprego = (
        ler_sidra('desemprego')
        .groupBy('localidade_id', 'periodo')
        .pivot('variavel_id', ['4099', '4118']).agg(F.first('valor'))
        .withColumnRenamed('4099', 'taxa_desocupacao_pct')
        .withColumnRenamed('4118', 'taxa_subutilizacao_pct')
        .withColumn('uf_id', F.col('localidade_id').cast('int'))
        .withColumn('ano', F.substring('periodo', 1, 4).cast('int'))
        .withColumn('trimestre_num', F.substring('periodo', 5, 2).cast('int'))
        .withColumn('trimestre', F.concat('trimestre_num', F.lit('º trimestre')))
        .withColumn('data_referencia', F.make_date('ano', (F.col('trimestre_num') - 1) * 3 + 1, F.lit(1)))
        .join(F.broadcast(dim_uf), 'uf_id', 'left')
        .select('data_referencia','ano','trimestre_num','trimestre','uf_id','uf_sigla',
            'uf_nome','regiao',F.col('taxa_desocupacao_pct').cast('decimal(10,2)'),F.col('taxa_subutilizacao_pct').cast('decimal(10,2)'),
        )
        .orderBy('data_referencia', 'uf_id')
    )
    fato_desemprego.show(10)
    fato_desemprego.printSchema()
    #salvar_csv(fato_desemprego, 'desemprego_uf_trimestre')

+---------------+----+-------------+------------+-----+--------+---------+--------+--------------------+----------------------+
|data_referencia| ano|trimestre_num|   trimestre|uf_id|uf_sigla|  uf_nome|  regiao|taxa_desocupacao_pct|taxa_subutilizacao_pct|
+---------------+----+-------------+------------+-----+--------+---------+--------+--------------------+----------------------+
|     2012-01-01|2012|            1|1º trimestre|   11|      RO| Rondônia|   Norte|                8.10|                 22.30|
|     2012-01-01|2012|            1|1º trimestre|   12|      AC|     Acre|   Norte|                9.20|                 28.30|
|     2012-01-01|2012|            1|1º trimestre|   13|      AM| Amazonas|   Norte|               11.10|                 23.50|
|     2012-01-01|2012|            1|1º trimestre|   14|      RR|  Roraima|   Norte|                8.50|                 22.30|
|     2012-01-01|2012|            1|1º trimestre|   15|      PA|     Pará|   Norte|                8.00|

In [18]:
# IPCA mensal por área pesquisada; não equivale ao total de cada UF.
if 'ipca' in INDICADORES_ATIVOS:
    fato_ipca = (
        ler_sidra('ipca')
        .withColumn('grupo_ipca', categoria(315))
        .groupBy('localidade_id', 'localidade_nome', 'nivel_id', 'periodo', 'grupo_ipca')
        .pivot('variavel_id', ['63', '69', '2265', '66']).agg(F.first('valor'))
        .withColumnRenamed('63', 'variacao_mensal_pct')
        .withColumnRenamed('69', 'acumulado_ano_pct')
        .withColumnRenamed('2265', 'acumulado_12_meses_pct')
        .withColumnRenamed('66', 'peso_mensal_pct')
        .withColumnRenamed('localidade_id', 'area_ipca_id')
        .withColumnRenamed('localidade_nome', 'area_ipca')
        .withColumn('ano', F.substring('periodo', 1, 4).cast('int'))
        .withColumn('mes', F.substring('periodo', 5, 2).cast('int'))
        .withColumn('data_referencia', F.make_date('ano', 'mes', F.lit(1)))
        .select(
            'data_referencia','ano','mes','area_ipca_id','area_ipca','nivel_id',
            'grupo_ipca','variacao_mensal_pct','acumulado_ano_pct',
            'acumulado_12_meses_pct','peso_mensal_pct',
        )
        .orderBy('data_referencia', 'area_ipca', 'grupo_ipca')
    )
    salvar_csv(fato_ipca, 'ipca_area_mes')

ipca_area_mes: 7,800 linhas -> C:\Users\Gabriel\Desktop\Python_Treinamento\IBGE\data\processed\ipca_area_mes.csv


In [19]:
arquivos_saida = sorted(PROCESSED_DIR.glob('*.csv'))
print('Pipeline concluído. Arquivos disponíveis para o Power BI:')
for arquivo in arquivos_saida:
    print(f'- {arquivo.name}: {arquivo.stat().st_size:,} bytes')

# Encerra recursos locais ao final de uma execução automatizada.
# Em análise interativa, mantenha comentado para continuar consultando os DataFrames.
# spark.stop()

Pipeline concluído. Arquivos disponíveis para o Power BI:
- desemprego_uf_trimestre.csv: 103,656 bytes
- dim_uf.csv: 734 bytes
- faixa_etaria_uf.csv: 130,521 bytes
- ipca_area_mes.csv: 654,086 bytes
- pib_brasil_ano.csv: 5,424 bytes
- populacao_municipio.csv: 4,545,344 bytes
- resumo_ano.csv: 490 bytes
